# Exercise 6 — UR5e geometry, estimation, and control

Work from a fresh kernel. The local DH model below is supplied so you can focus
on derivatives, identification, estimation, and optimization. Implement each
task in the marked cells; inspect [the complete solution](../solutions/06_ur5e.ipynb)
after attempting it. Use deterministic seeds and separate plant truth from fitted models.

### Prepare the notebook and optional interactive frontend

Imports below provide array operations, CasADi graphs, and presentation only.
Every robot model and optimization formulation is defined in this notebook.
Plotly gives hover/zoom plots when installed; Matplotlib is the fallback.
Widgets use explicit **Run** and **Reset** buttons, so moving a slider cannot
launch a solve. `CASADI_TUTORIAL_WIDGETS=0` disables widgets for batch execution.

In [ ]:
import os
import sys
from pathlib import Path
from time import perf_counter

repository_root = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "casadi_tutorial").is_dir()
)
if str(repository_root) not in sys.path:
    sys.path.insert(0, str(repository_root))

import casadi as ca
import matplotlib.pyplot as plt
import numpy as np
from IPython.display import display

from casadi_tutorial.interactive import experiment
from casadi_tutorial.plotting import display_plot, plot_robot, plot_series
from casadi_tutorial.diagnostics import bound_report, max_bound_violation, solver_summary

try:
    import ipywidgets as widgets
except ImportError:
    widgets = None

np.set_printoptions(precision=5, suppress=True)

### Write the robot model: frames, parameters, and coordinates

The table below is the [UR5e standard DH geometry published by Universal Robots](https://www.universal-robots.com/articles/ur/application-installation/dh-parameters-for-calculations-of-kinematics-and-dynamics/).
Rows follow joints 1–6 from base to tool; all lengths are metres and all angles are radians.

| Joint | $a_i$ | $d_i$ | $\alpha_i$ |
|---|---:|---:|---:|
| 1 | 0 | 0.1625 | $\pi/2$ |
| 2 | −0.425 | 0 | 0 |
| 3 | −0.3922 | 0 | 0 |
| 4 | 0 | 0.1333 | $\pi/2$ |
| 5 | 0 | 0.0997 | $-\pi/2$ |
| 6 | 0 | 0.0996 | 0 |

For **standard** DH, the relative transform is

$$
{}^{i-1}T_i=\begin{bmatrix}
c_i&-s_i c_{\alpha_i}&s_i s_{\alpha_i}&a_i c_i\\
s_i&c_i c_{\alpha_i}&-c_i s_{\alpha_i}&a_i s_i\\
0&s_{\alpha_i}&c_{\alpha_i}&d_i\\0&0&0&1
\end{bmatrix},\qquad
T^0_6=\prod_{i=1}^6{}^{i-1}T_i=\begin{bmatrix}R&p\\0&1\end{bmatrix}.
$$

Calibration will adjust only $\theta=[\Delta a_2,\Delta a_3,\delta_2,\delta_3]$.
Here $q$ always means the six **encoder coordinates**, and
$q_{\rm physical}=q+[0,\delta_2,\delta_3,0,0,0]^T$.
The offset is added exactly once, inside this model. Fixed base and tool frames are assumed known.
`pose_map(q, theta)` returns $4\times4$; `position_map` returns $3\times1$;
`origins_map` returns $3\times7$. Joint, speed, and acceleration limits below are conservative teaching choices.
These functions are defined locally so the whole graph can be inspected and edited here.

In [ ]:
# Standard DH, metres and radians. The table is included here rather than imported.
dh_a = np.array([0.0, -0.425, -0.3922, 0.0, 0.0, 0.0])
dh_d = np.array([0.1625, 0.0, 0.0, 0.1333, 0.0997, 0.0996])
dh_alpha = np.array([np.pi / 2, 0.0, 0.0, np.pi / 2, -np.pi / 2, 0.0])
q_min_physical = np.full(6, -2 * np.pi)
q_max_physical = np.full(6, 2 * np.pi)
v_max = np.full(6, np.pi)
u_max = np.full(6, 2 * np.pi)


def encoder_bias(theta):
    """theta = [delta_a2, delta_a3, delta_encoder2, delta_encoder3]."""
    return ca.vertcat(0, theta[2], theta[3], 0, 0, 0)


def dh_transform(angle, distance, length, twist):
    """One STANDARD-DH transform RotZ(angle) TransZ(d) TransX(a) RotX(alpha)."""
    c, s = ca.cos(angle), ca.sin(angle)
    ct, st = np.cos(twist), np.sin(twist)
    return ca.vertcat(
        ca.horzcat(c, -s * ct, s * st, length * c),
        ca.horzcat(s, c * ct, -c * st, length * s),
        ca.horzcat(0, st, ct, distance),
        ca.horzcat(0, 0, 0, 1),
    )


def link_transforms(q_encoder, theta=None):
    """Seven base-to-frame transforms, including the identity base frame."""
    theta = ca.DM.zeros(4) if theta is None else theta
    physical_q = q_encoder + encoder_bias(theta)
    lengths = ca.vertcat(dh_a[0], dh_a[1] + theta[0], dh_a[2] + theta[1], *dh_a[3:])
    transforms = [ca.DM.eye(4)]
    for joint in range(6):
        transforms.append(
            transforms[-1] @ dh_transform(
                physical_q[joint], dh_d[joint], lengths[joint], dh_alpha[joint]
            )
        )
    return transforms


def forward_kinematics(q_encoder, theta=None):
    return link_transforms(q_encoder, theta)[-1]


def frame_origins(q_encoder, theta=None):
    return ca.horzcat(*[transform[:3, 3] for transform in link_transforms(q_encoder, theta)])


q_symbol = ca.SX.sym("q_encoder", 6)
theta_symbol = ca.SX.sym("theta", 4)
pose_map = ca.Function("pose", [q_symbol, theta_symbol], [forward_kinematics(q_symbol, theta_symbol)])
position_map = ca.Function("position", [q_symbol, theta_symbol], [pose_map(q_symbol, theta_symbol)[:3, 3]])
origins_map = ca.Function("origins", [q_symbol, theta_symbol], [frame_origins(q_symbol, theta_symbol)])
nominal_theta = np.zeros(4)

### Task A — derive and validate both Jacobian blocks

Assemble $J_i=[z_{i-1}\times(p_6-p_{i-1});z_{i-1}]$. Validate its upper block
against CasADi AD, then check linear and angular directional derivatives at
the zero pose and five random configurations. Verify $R^TR=I$ and $\det R=1$.
State why joint 6 changes orientation but not the tool-origin position.
**Acceptance:** derivative error below $2\times10^{-8}$, correct dimensions,
and an explanation of zero-pose singularity.

In [ ]:
# TODO: define geometric_jacobian and compare both blocks with finite differences.
# Start with transforms = link_transforms(q_symbol, nominal_theta).
# Store the result in a ca.Function so you can evaluate several configurations.

### Create independent pose measurements and reserve validation data

A synthetic tracker observes the complete tool pose at 36 diverse, known encoder
configurations. The simulated plant has 4 mm and −3 mm link corrections and small
joint-2/3 zero offsets. **Only data generation sees `theta_truth`.** The estimator
starts at nominal parameters. Tracker noise is 0.5 mm per Cartesian component
and 0.002 rad per rotation-vector component.

Twenty-four randomly selected poses train the estimator; the remaining twelve
are held out. The split and random seed stay fixed when changing noise, allowing
fair comparisons. Tool/base extrinsics, other DH constants, and encoder noise
during this static calibration experiment are assumed known. Estimating every
DH entry and frame offset together would introduce ambiguity.

In [ ]:
from scipy.spatial.transform import Rotation

parameter_scale = np.array([0.01, 0.01, 0.05, 0.05])
theta_truth = np.array([0.004, -0.003, 0.015, -0.012])
sigma_position = 0.0005
sigma_rotation = 0.002


def calibration_dataset(noise_multiplier=1.0):
    rng = np.random.default_rng(2026)
    configurations = rng.uniform(
        [-1.5, -1.5, 0.2, -1.0, 0.4, -1.5],
        [1.5, -0.2, 1.7, 1.0, 1.6, 1.5], (36, 6),
    )
    measured = np.array([np.asarray(pose_map(q, theta_truth)) for q in configurations])
    measured[:, :3, 3] += rng.normal(0, sigma_position * noise_multiplier, (36, 3))
    for transform in measured:
        rotation_noise = Rotation.from_rotvec(rng.normal(0, sigma_rotation * noise_multiplier, 3)).as_matrix()
        transform[:3, :3] = transform[:3, :3] @ rotation_noise
    indices = rng.permutation(36)
    return {
        "q": configurations, "poses": measured,
        "training": indices[:24], "validation": indices[24:],
        "noise_multiplier": noise_multiplier,
    }


calibration_data = calibration_dataset()
print("Training / validation poses:", len(calibration_data["training"]), "/", len(calibration_data["validation"]))

### Task B — calibrate four identifiable corrections

Fit $\theta=[\Delta a_2,\Delta a_3,\delta_2,\delta_3]$ using the 24 training
poses and a position-plus-chordal-rotation residual. Scale parameters by
`[0.01, 0.01, 0.05, 0.05]`, constrain normalized corrections to `[−1,1]`, and
keep all twelve validation poses out of the objective. Report separate
held-out position and orientation RMSE and the scaled sensitivity singular values.
**Acceptance:** full sensitivity rank, held-out position RMSE below nominal
and below 2 mm. Explain why arbitrary DH/base/tool calibration can be ambiguous.

In [ ]:
# TODO: create the scaled Opti least-squares problem, fit theta_estimate,
# then score nominal and fitted geometry on data["validation"] only.

### Task C — estimate joint velocity instead of differentiating noisy encoders

Derive $A_d,B_d$ for $x=[q;v]$ with held acceleration. Build a six-interval
MHE with encoder and tool-position observations, acceleration disturbances,
and an arrival prior. Use fitted FK in the tool observation; keep encoder
observations in encoder coordinates. Generate an independent disturbed plant.
**Acceptance:** finite estimates, small dynamics defects, and velocity RMSE
below a backward-difference baseline after window warm-up. Plot truth,
measurements, estimates, and residuals for a selected joint.

In [ ]:
# TODO: define exact_step, build_mhe, simulate measurements, and estimate.
# Use X[:6, :] for all position rows; X[:6] flattens a CasADi matrix.

### Task D — solve a complete obstacle-avoiding OCP on two meshes

Plan the rest-to-rest transfer in chapter 17 with $N=15$ and $N=24$ over 3 s.
Include position, velocity, acceleration, endpoint, exact dynamics, and soft
spherical **tool-point** constraints. Place a 7 cm sphere on the nominal tool
path and use a 1.5 cm planning margin. Explain dimensionless slack and use
time-weighted costs so meshes solve comparable problems.

Warm-start the fine mesh from the coarse solution; compare objective,
constraint-Jacobian dimensions/nonzeros/density, and dense true clearance.
**Acceptance:** both solves succeed, endpoint/dynamics/limit checks pass,
and actual dense clearance is positive independently of slack. Plot $q(t)$,
$v(t)$, held $u(t)$, 3D motion, and clearance for both meshes.

In [ ]:
# TODO: implement the full Opti OCP, solve both meshes, and reconstruct
# q(tau) = q_k + tau*v_k + 0.5*tau**2*u_k between nodes.
# Record Jacobian sparsity using ca.jacobian(opti.g, opti.x).

### Task E — close the estimation/control loop and diagnose a failure

Track the coarse reference with a 12-step MPC for 24 feedback moves. Feed its
initial state from MHE; apply only the first input to a separate disturbed
plant, measure again, and shift warm starts. Use terminal penalties rather
than a hard terminal equality at every move. Show final true target error,
sampled true clearance, solver statuses, and commanded-input limits.

Then choose an infeasible duration or an inaccurate geometry assumption.
Explain the failure using reachability bounds, residuals, and solver diagnostics;
show a concrete repair. **Acceptance:** default feedback has positive sampled
tool clearance and final Cartesian target error below 3 cm. Distinguish this
numerical demonstration from torque-level or certified collision-safe control.

In [ ]:
# TODO: implement the parameterized MPC and the sensor -> MHE -> MPC loop.
# Keep model truth in simulation/scoring; never pass it as the MPC initial state.